# World Happiness 2015–2019: Drivers, Disparities, and Statistical Tests

_A reproducible enquiry into what moves the World Happiness Score across countries and years, enriched with World Bank and REST Countries data._

### Contents

1. [Introduction](#1.-Introduction)
2. [Data Sources & Acquisition](#2.-Data-Sources-&-Acquisition)
3. [Data Wrangling Pipeline](#3.-Data-Wrangling-Pipeline)
    - 3.1 [Schema reconciliation across years](#3.1-Schema-reconciliation-across-years)
    - 3.2 [Country-name normalization (ISO-3)](#3.2-Country-name-normalization-(ISO-3))
    - 3.3 [Enrichment from World Bank](#3.3-Enrichment-from-World-Bank)
    - 3.4 [Enrichment from REST Countries](#3.4-Enrichment-from-REST-Countries)
    - 3.5 [Missing values & outliers](#3.5-Missing-values-&-outliers)
4. [Exploratory Analysis & Visualization](#4.-Exploratory-Analysis-&-Visualization)
    - 4.1 [Global distribution (2019)](#4.1-Global-distribution-(2019))
    - 4.2 [Regional comparison](#4.2-Regional-comparison)
    - 4.3 [Time evolution 2015 to 2019](#4.3-Time-evolution-2015-to-2019)
    - 4.4 [Correlation matrix](#4.4-Correlation-matrix)
5. [Hypothesis Testing](#5.-Hypothesis-Testing)
    - 5.1 [H1 — Western Europe vs Sub-Saharan Africa (independent t-test)](#5.1-H1-—-Western-Europe-vs-Sub-Saharan-Africa)
    - 5.2 [H2 — GDP per capita predicts happiness (Pearson r)](#5.2-H2-—-GDP-per-capita-predicts-happiness)
    - 5.3 [H3 — Income group vs happiness class (chi-squared)](#5.3-H3-—-Income-group-vs-happiness-class)
6. [Conclusions](#6.-Conclusions)
7. [Summary](#7.-Summary)
8. [References](#8.-References)

## 1. Introduction

Every spring the **World Happiness Report** publishes a single numeric **happiness score** per country, derived from the Gallup World Poll's Cantril ladder question ("Imagine a ladder, with steps numbered from 0 at the bottom to 10 at the top — the top represents the best possible life for you…"). Alongside the score the report attributes the country's position to six contributing factors: GDP per capita, social support, healthy life expectancy, freedom to make life choices, generosity, and perceptions of corruption.

While the report itself summarises results, it stops short of running formal statistical tests across the five-year window 2015–2019. This notebook closes that gap and asks three concrete questions:

- **H1.** Does the average happiness score in **Western Europe** differ significantly from that in **Sub-Saharan Africa** (independent two-sample t-test)?
- **H2.** Is national happiness linearly associated with **GDP per capita PPP** when measured against an external source — the World Bank — instead of the WHR's internal estimate (Pearson correlation)?
- **H3.** Is a country's **World Bank income group** (Low / Lower-middle / Upper-middle / High) **independent** of its happiness class (above vs. below the global median) (chi-squared test of independence)?

All hypothesis tests use $\alpha = 0.05$. The pipeline that produces the analysis-ready table is documented in section 3 and lives entirely under [`src/`](./src), so the notebook itself stays narrative and short. To re-run end to end, execute every cell top-to-bottom inside the project's Docker image.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data import happiness, reconcile, restcountries, worldbank, pipeline
from src.charts import distributions, relationships, geographic, timeseries

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 140)

## 2. Data Sources & Acquisition

Three independent sources feed the analysis. Two of them are remote APIs, both queried at notebook start and cached on disk so subsequent runs work offline.

| # | Source | Coverage | Access | Cache location |
|---|---|---|---|---|
| 1 | World Happiness Report 2015–2019 (Kaggle CSV dump) | One CSV per year, 155–158 countries each | Local files | `data/raw/world-happines-2015-2019/` |
| 2 | World Bank Open Data | GDP per capita PPP, unemployment, income group | `wbgapi` Python client | `data/raw/worldbank/` |
| 3 | REST Countries v3.1 | Region, subregion, population | `requests.get('https://restcountries.com/v3.1/all')` | `data/raw/restcountries/` |

We start by inspecting one raw WHR CSV to motivate the schema drift discussion in §3.1.

In [2]:
raw_dir = Path('data/raw/world-happines-2015-2019')
raw_2015 = pd.read_csv(raw_dir / '2015.csv')
raw_2018 = pd.read_csv(raw_dir / '2018.csv')

print('2015 columns:', list(raw_2015.columns))
print('2018 columns:', list(raw_2018.columns))
print('2015 shape:', raw_2015.shape, ' 2018 shape:', raw_2018.shape)

2015 columns: ['Country', 'Region', 'Happiness Rank', 'Happiness Score', 'Standard Error', 'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)', 'Generosity', 'Dystopia Residual']
2018 columns: ['Overall rank', 'Country or region', 'Score', 'GDP per capita', 'Social support', 'Healthy life expectancy', 'Freedom to make life choices', 'Generosity', 'Perceptions of corruption']
2015 shape: (158, 12)  2018 shape: (156, 9)


## 3. Data Wrangling Pipeline

The end-to-end build is encapsulated in `pipeline.build_dataset()`. The next five subsections walk through what it does, in the same order, with intermediate frames so the transformation is auditable.

```text
raw 5 yearly CSVs ----+
                      |
                      v
         happiness.load_all  ---+
                                |
         reconcile.normalize_names
                                |
 worldbank.fetch_*  ----------> + <- restcountries.fetch_country_metadata
                                |
                                v
             data/processed/happiness_enriched.csv
```

### 3.1 Schema reconciliation across years

The five CSVs use **four different naming conventions**:

- 2015 — `Happiness Score`, `Economy (GDP per Capita)`, `Family`, `Health (Life Expectancy)`, `Trust (Government Corruption)`…
- 2016 — same as 2015 but `Standard Error` becomes a confidence-interval pair
- 2017 — dot-separated names (`Happiness.Score`, `Economy..GDP.per.Capita.`, `Whisker.high`)
- 2018 / 2019 — short modern names (`Score`, `GDP per capita`, `Social support`, `Healthy life expectancy`)

`happiness.load_year(year)` knows the per-year mapping and returns a frame in the canonical schema `(country, year, score, gdp, social_support, life_expectancy, freedom, generosity, corruption)`. `happiness.load_all()` concatenates the five frames into a single long table.

In [3]:
happiness_long = happiness.load_all()
print('long-format shape:', happiness_long.shape)
print('rows per year:')
print(happiness_long.groupby('year').size().to_string())
happiness_long.head()

long-format shape: (782, 9)
rows per year:
year
2015    158
2016    157
2017    155
2018    156
2019    156


,country,year,score,gdp,social_support,life_expectancy,freedom,generosity,corruption
0,Afghanistan,2015,3.575,0.31982,0.30285,0.30335,0.23414,0.36510,0.09719
1,Albania,2015,4.959,0.87867,0.80434,0.81325,0.35733,0.14272,0.06413
2,Algeria,2015,5.605,0.93929,1.07772,0.61766,0.28579,0.07822,0.17383
3,Angola,2015,4.033,0.75778,0.86040,0.16683,0.10384,0.12344,0.07122
4,Argentina,2015,6.574,1.05351,1.24823,0.78723,0.44974,0.11451,0.08484


### 3.2 Country-name normalization (ISO-3)

Names diverge between sources (`United States` vs `United States of America`, `Hong Kong S.A.R., China` vs `Hong Kong`, `Czech Republic` vs `Czechia`). To dodge those landmines every join in the pipeline happens on the **ISO 3166-1 alpha-3 code**, produced by `reconcile.normalize_names`, which delegates to the `country_converter` library and falls back to a manual `OVERRIDES` dict if needed.

Verification: every name in the WHR corpus must resolve. Anything that doesn't would surface as `NaN` in `country_iso3`.

In [4]:
happiness_iso = reconcile.normalize_names(happiness_long, col='country')
n_unresolved = happiness_iso['country_iso3'].isna().sum()
print(f'rows: {len(happiness_iso)}, unresolved country names: {n_unresolved}')
happiness_iso[['country', 'country_iso3', 'year', 'score']].head()

rows: 782, unresolved country names: 0


,country,country_iso3,year,score
0,Afghanistan,AFG,2015,3.575
1,Albania,ALB,2015,4.959
2,Algeria,DZA,2015,5.605
3,Angola,AGO,2015,4.033
4,Argentina,ARG,2015,6.574


### 3.3 Enrichment from World Bank

From the World Bank we pull two indicators and one categorical attribute:

- `NY.GDP.PCAP.PP.CD` — **GDP per capita, PPP** (current international $). Comparable across countries; complements the WHR's internal `gdp` factor (which is already a log-transformed contribution to the score, not the raw figure).
- `SL.UEM.TOTL.ZS` — **Unemployment**, total (% of total labor force, ILO modelled estimate).
- **Income group** — World Bank classification (`Low income`, `Lower middle income`, `Upper middle income`, `High income`). Required for H3.

All three calls are cached as CSVs under `data/raw/worldbank/`.

In [5]:
wb_gdp = worldbank.fetch_gdp_per_capita_ppp(years=range(2015, 2020))
wb_unemp = worldbank.fetch_unemployment(years=range(2015, 2020))
wb_income = worldbank.fetch_income_groups()

print('GDP PPP rows:', len(wb_gdp), ' unemployment rows:', len(wb_unemp), ' income groups:', len(wb_income))
print('income group counts:')
print(wb_income['income_group'].value_counts().to_string())

GDP PPP rows: 1330  unemployment rows: 1330  income groups: 215
income group counts:
income_group
High income            86
Upper middle income    54
Lower middle income    50
Low income             25


### 3.4 Enrichment from REST Countries

REST Countries gives us the **region / subregion / population** that the WHR drops from 2017 onwards. The query goes against `https://restcountries.com/v3.1/all?fields=name,cca3,region,subregion,population` and is cached as a single JSON blob.

It also supplies the `subregion` column we need to slice Western Europe and Sub-Saharan Africa for **H1**.

In [6]:
rc_meta = restcountries.fetch_country_metadata()
print('REST Countries rows:', len(rc_meta))
print('regions:')
print(rc_meta['region'].value_counts().to_string())
rc_meta.head()

REST Countries rows: 250
regions:
region
Africa       59
Americas     56
Europe       53
Asia         50
Oceania      27
Antarctic     5


,country_iso3,name_common,region,subregion,population
0,ABW,Aruba,Americas,Caribbean,107566
1,AFG,Afghanistan,Asia,Southern Asia,43844000
2,AGO,Angola,Africa,Middle Africa,36170961
3,AIA,Anguilla,Americas,Caribbean,16010
4,ALA,Åland Islands,Europe,Northern Europe,30654


### 3.5 Missing values & outliers

`pipeline.build_dataset()` performs the four left joins (happiness ⊓ GDP ⊓ unemployment ⊓ income group ⊓ REST Countries metadata) and writes the result to `data/processed/happiness_enriched.csv`.

Left joins keep every WHR row even when a complementary source has no entry for that country. We expect a handful of missing values for:

- **Taiwan and Palestinian Territories** — not present in the World Bank classification.
- **Recent micro-states** — occasionally missing from REST Countries.
- **Unemployment** — some Gulf states and small island nations are not in the ILO modelled estimates.

We **do not impute** the missing values: each hypothesis test below filters its inputs with `dropna` so that exclusions are explicit and reproducible.

In [7]:
df = pipeline.build_dataset(use_cache=True)
print('enriched shape:', df.shape)
print('\nNaNs per column:')
print(df.isna().sum().sort_values(ascending=False).to_string())

enriched shape: (782, 17)

NaNs per column:
gdp_ppp            21
income_group       15
unemployment       10
population          5
subregion           5
region              5
name_common         5
corruption          1
year                0
country_iso3        0
generosity          0
freedom             0
life_expectancy     0
social_support      0
gdp                 0
score               0
country             0


In [8]:
# Outlier sanity check: countries whose 2019 score is more than 2 SD from the global mean
scores_2019 = df.query('year == 2019')['score']
mu, sigma = scores_2019.mean(), scores_2019.std(ddof=1)
outliers = df.query('year == 2019').assign(z=(df.query('year == 2019')['score'] - mu) / sigma)
outliers = outliers[outliers['z'].abs() > 2][['country', 'score', 'z', 'region']].sort_values('score')
print(f'2019 mean = {mu:.3f}, sd = {sigma:.3f}')
outliers

2019 mean = 5.407, sd = 1.113


,country,score,z,region
754,South Sudan,2.853,-2.294538,Africa
650,Central African Republic,3.083,-2.087912,Africa
669,Finland,7.769,2.121877,Europe


## 4. Exploratory Analysis & Visualization

### 4.1 Global distribution (2019)

_TODO: describe overall shape, mention skewness, call out top/bottom outliers._

In [9]:
# distributions.histogram_score(df, year=2019)

### 4.2 Regional comparison

_TODO: discuss inter-region spread and within-region variance._

In [10]:
# distributions.boxplot_by_region(df, year=2019)
# geographic.bar_mean_by_region(df, year=2019)

### 4.3 Time evolution 2015 to 2019

_TODO: comment on regional trends and notable movers._

In [11]:
# timeseries.lineplot_score_over_years(df, by='region')

### 4.4 Correlation matrix

_TODO: highlight strongest pairwise correlations, foreshadow H2._

In [12]:
# relationships.correlation_heatmap(df)

## 5. Hypothesis Testing

All tests use $\alpha = 0.05$ unless stated otherwise.

### 5.1 H1 — Western Europe vs Sub-Saharan Africa

**Test:** independent two-sample t-test (Welch's, unequal variances).

- $H_0$: mean happiness scores are equal across the two regions.
- $H_1$: mean happiness scores differ across the two regions.
- $\alpha$: 0.05.

_TODO: rationale, assumption checks (normality / variance)._

In [13]:
# group_we = df.query("region == 'Europe' and subregion == 'Western Europe' and year == 2019")['score']
# group_ssa = df.query("region == 'Africa' and subregion == 'Sub-Saharan Africa' and year == 2019")['score']
# stats.ttest_ind(group_we, group_ssa, equal_var=False)

In [14]:
# distributions.violin_score_by_region(df.query("subregion in ['Western Europe', 'Sub-Saharan Africa']"), year=2019)

_TODO: report t-statistic, p-value, decision, plain-English interpretation._

### 5.2 H2 — GDP per capita predicts happiness

**Test:** Pearson product-moment correlation (with associated p-value).

- $H_0$: $\rho = 0$ (no linear relationship between GDP per capita PPP and happiness score).
- $H_1$: $\rho \neq 0$.
- $\alpha$: 0.05.

_TODO: rationale, log-transform discussion if used._

In [15]:
# subset = df.query('year == 2019').dropna(subset=['gdp_ppp', 'score'])
# stats.pearsonr(subset['gdp_ppp'], subset['score'])

In [16]:
# relationships.scatter_with_regression(df, x='gdp_ppp', y='score', year=2019)

_TODO: report r, p-value, decision, plain-English interpretation._

### 5.3 H3 — Income group vs happiness class

**Test:** Pearson chi-squared test of independence on a 4 × 2 contingency table (income group × above/below median happiness).

- $H_0$: World Bank income group and happiness class are independent.
- $H_1$: World Bank income group and happiness class are dependent.
- $\alpha$: 0.05.

_TODO: rationale, expected-frequency assumption check._

In [17]:
# subset = df.query('year == 2019').dropna(subset=['income_group', 'score'])
# subset['happiness_class'] = (subset['score'] > subset['score'].median()).map({True: 'high', False: 'low'})
# table = pd.crosstab(subset['income_group'], subset['happiness_class'])
# stats.chi2_contingency(table)

In [18]:
# TODO: stacked bar of happiness_class proportions per income_group

_TODO: report chi-squared, p-value, degrees of freedom, decision, plain-English interpretation._

## 6. Conclusions

_TODO: synthesize the three test outcomes and what they collectively say about the drivers of national happiness._

## 7. Summary

_TODO: TL;DR of the entire study (3-4 bullet points)._

## 8. References

1. Helliwell, J. F., Layard, R., Sachs, J. D. _World Happiness Report_ 2015–2019 editions. https://worldhappiness.report/data/
2. World Bank. _World Development Indicators_. https://data.worldbank.org/
3. REST Countries. https://restcountries.com/
4. SciPy `scipy.stats` documentation. https://docs.scipy.org/doc/scipy/reference/stats.html
5. McKinney, W. _Python for Data Analysis_. O'Reilly, 3rd ed., 2022.